### **1. Import, Load, Clean**

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
# Load dataset
PATH = "../data/raw/listings.csv"
df = pd.read_csv(PATH)

# Price cleaning
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Percentage cleaning
rates = ["host_response_rate", "host_acceptance_rate"]
for col in rates:
    df[col] = df[col].astype(str).str.replace("%", "").astype(float) / 100

# Remove rows where price is missing
df = df.dropna(subset=["price"]).copy()

In [3]:
cols_to_drop = [

    # identifiers / urls / metadata
    "id",
    "listing_url",
    "scrape_id",
    "last_scraped",
    "source",
    "picture_url",
    "host_id",
    "host_url",
    "host_thumbnail_url",
    "host_picture_url",
    "calendar_updated",
    "calendar_last_scraped",

    # leakage
    "estimated_revenue_l365d",
    "estimated_occupancy_l365d",

    # text fields (no NLP)
    "name",
    "description",
    "neighborhood_overview",
    "host_about",

    # redundant text versions
    "bathrooms_text",
    "host_name",
    "host_verifications",

    # review score redundancy
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",

    # host listing redundancy
    "host_listings_count",
    "host_total_listings_count",

    # availability redundancy
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_eoy",

    # review activity redundancy
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",

    # derived night statistics
    "minimum_minimum_nights",
    "maximum_minimum_nights",
    "minimum_maximum_nights",
    "maximum_maximum_nights",
    "minimum_nights_avg_ntm",
    "maximum_nights_avg_ntm",

    # categorical removal from EDA
    "host_since",
    "first_review",
    "last_review",
    "amenities",
    "license",
    "host_location",
    "host_neighbourhood",
    "neighbourhood",
    "neighbourhood_cleansed",

    # weak categorical predictor
    "host_response_time"
]

In [4]:
df["price_bin"] = pd.qcut(df["price"], q=5, duplicates="drop")

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["price_bin"]
)

df_train = df_train.drop(columns=["price_bin"])
df_test = df_test.drop(columns=["price_bin"])

In [5]:
df_train["log_price"] = np.log1p(df_train["price"])
df_test["log_price"] = np.log1p(df_test["price"])

In [6]:
X_train = df_train.drop(["price", "log_price"], axis=1)
y_train = df_train["log_price"]

X_test = df_test.drop(["price", "log_price"], axis=1)
y_test = df_test["log_price"]

In [7]:
X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
X_test = X_test.drop(columns=cols_to_drop, errors="ignore")

In [8]:
print(X_train.shape)
print(X_test.shape)

(4520, 26)
(1131, 26)


### **2. Handling missing values**

In [9]:
missing = X_train.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0]

review_scores_rating      9.336283
reviews_per_month         9.336283
host_response_rate        7.743363
host_acceptance_rate      4.823009
host_is_superhost         2.898230
has_availability          0.575221
beds                      0.243363
bathrooms                 0.088496
host_identity_verified    0.044248
host_has_profile_pic      0.044248
bedrooms                  0.044248
dtype: float64

In [10]:
from sklearn.impute import SimpleImputer

num_cols = X_train.select_dtypes(include="number").columns

num_imputer = SimpleImputer(strategy="median")

X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

In [11]:
cat_cols = X_train.select_dtypes(include="object").columns

cat_imputer = SimpleImputer(strategy="most_frequent")

X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_66654/1189760543.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns


In [12]:
X_train.isna().sum().sum(), X_test.isna().sum().sum()

(np.int64(0), np.int64(0))

### **3. Transformations**

In [13]:
BILBAO_CENTER_LAT = 43.2630
BILBAO_CENTER_LON = -2.9350

def distance_to_center(lat, lon):
    return np.sqrt((lat - BILBAO_CENTER_LAT)**2 + (lon - BILBAO_CENTER_LON)**2)

In [14]:
X_train["distance_to_center"] = distance_to_center(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_center"] = distance_to_center(
    X_test["latitude"], X_test["longitude"]
)

In [15]:
clip_cols = [
    "beds",
    "minimum_nights",
    "maximum_nights"
]

for col in clip_cols:
    
    lower = X_train[col].quantile(0.01)
    upper = X_train[col].quantile(0.99)

    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

In [16]:
skewed = [
    "minimum_nights",
    "maximum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count"
]

for col in skewed:
    X_train[col] = np.log1p(X_train[col])
    X_test[col] = np.log1p(X_test[col])

In [17]:
cat_cols = X_train.select_dtypes(include="object").columns
cat_cols

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_66654/1847301736.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns


Index(['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
       'neighbourhood_group_cleansed', 'property_type', 'room_type',
       'has_availability', 'instant_bookable'],
      dtype='str')

In [18]:
binary_cols = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "has_availability",
    "instant_bookable"
]

for col in binary_cols:
    X_train[col] = X_train[col].map({"t":1, "f":0})
    X_test[col] = X_test[col].map({"t":1, "f":0})

In [19]:
TOP_K = 10

top_properties = (
    X_train["property_type"]
    .value_counts()
    .nlargest(TOP_K)
    .index
)

X_train["property_type_clean"] = X_train["property_type"].where(
    X_train["property_type"].isin(top_properties),
    "Other"
)

X_test["property_type_clean"] = X_test["property_type"].where(
    X_test["property_type"].isin(top_properties),
    "Other"
)

X_train = X_train.drop(columns=["property_type"])
X_test = X_test.drop(columns=["property_type"])

In [20]:
categorical_cols = [
    "property_type_clean",
    "room_type",
    "neighbourhood_group_cleansed"
]

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)

model.fit(X_train_scaled, y_train)